In [ ]:
import numpy as np
import pandas as pd
import tensorflow_hub as hub
import tensorflow as tf
# import tensorflow_text
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

In [ ]:
df = pd.read_csv('data/weighted_score_above_08.csv')

In [ ]:
df.info()

In [ ]:
def clean(texts):
    return texts.strip()

In [ ]:
''' 
SBERT can capture semantic relationships and support multilingual data,
it has high quality embedding but the embedding process may took time. 
The feature space of SBERT is 384 and has dense results.
'''

In [ ]:
texts = df['review'].astype(str).apply(clean).tolist()
sbert_mul = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device='cuda')

batch_size, l = 512, len(texts)
embedded_sbert_mul = []
for i in tqdm(range(0, l, batch_size), desc='Embedding using multilingual SBERT'):
    batch = texts[i:min(l, i+batch_size)]
    emb_sbert_mul = sbert_mul.encode(
        batch,
        show_progress_bar=False,
        convert_to_numpy=True
    )
    embedded_sbert_mul.append(emb_sbert_mul)

In [ ]:
embedded_sbert_mul = pd.DataFrame(np.vstack(embedded_sbert_mul), columns=[f'emb{i}' for i in range(384)])
df_sbert_mul = pd.concat([df.iloc[:, :12], embedded_sbert_mul, df.iloc[:, 12:]], axis=1)

In [ ]:
df_sbert_mul.to_csv('results/df_sbert_mul.csv', index=False)

In [ ]:
'''
Universal Sentence Encoder can capture semantic relationships and also support multilingual datasets like SBERT.
It has lower quality embeddings but faster speed than SBERT.
The feature space of it is 512.
'''

In [ ]:
texts = df['review'].astype(str).apply(clean).tolist()
# Requires tensorflow_text but failed to pip install it in Python 3.11.5
use_mul = hub.load('https://tfhub.dev/google/universal-sentence-encoder-multilingual/3')

batch_size, l = 512, len(texts)
embedded_use_mul = []
for i in tqdm(range(0, l, batch_size), desc='Embedding using multilingual USE'):
    batch = texts[i:min(l, i+batch_size)]
    emb_use_mul = use_mul(batch)
    embedded_use_mul.append(emb_use_mul)

In [ ]:
embedded_use_mul = pd.DataFrame(np.vstack(embedded_use_mul), columns=[f'emb{i}' for i in range(512)])
df_use_mul = pd.concat([df.iloc[:, :12], embedded_use_mul, df.iloc[:, 12:]], axis=1)

In [ ]:
df_use_mul.to_csv('results/df_use_mul.csv', index=False)